# 01 · Index audit and splits

## Purpose
Check that the indexed MAFAULDA tree matches the published layout, audit shaft speed and recording quality, and
confirm the frozen recording-level splits are balanced. Nothing here trains a model or looks at test labels to
make a choice: splits were assigned by `scripts/02_split.py` from class and speed alone.

## Inputs
The committed `metadata.csv` (from `01_index.py`, `00_convert.py`, `02_split.py`), the resolved config and its hashes, and `data/raw_npy/` for the example traces.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml.config import config_sha256, load_config
from ml.labels import CLASS10
from ml.paths import DEFAULT_CONFIG, SYSTEM_CODE_DIR, data_dir
from ml.preprocess import tacho_intervals
from ml.splits import FAULT_CLASSES, split_hash

cfg = load_config(DEFAULT_CONFIG)
pre = cfg.preprocess
meta = pd.read_csv(SYSTEM_CODE_DIR / "metadata.csv")
FIG_DIR = SYSTEM_CODE_DIR / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

print(f"preprocess config sha256 {config_sha256(pre)[:16]}…")
print(f"split hash               {split_hash(meta)[:16]}…")
print(f"metadata.csv             {meta.shape[0]} rows × {meta.shape[1]} columns")

### Counts against the published table
Every class count must equal CLAUDE.md §3; `01_index.py` already refuses to write otherwise, so this is the visible record.

In [ ]:
counts = pd.DataFrame({
    "found": meta["class10"].value_counts().reindex(CLASS10),
    "expected": pd.Series(pre.expected_class_counts).reindex(CLASS10),
})
counts["match"] = counts["found"] == counts["expected"]
counts.loc["total"] = [counts["found"].sum(), counts["expected"].sum(), counts["match"].all()]
counts

Shape, NaN and read problems across all files (expected: none).

In [ ]:
pd.Series({
    "files not 250000 × 8": int(((meta.n_rows != pre.n_rows) | (meta.n_cols != pre.n_channels)).sum()),
    "files with NaN": int((meta.nan_count > 0).sum()),
    "npy verified": int((meta.n_rows_verified == pre.n_rows).sum()),
})

### Shaft speed: measured vs nominal
The file name is the nominal shaft frequency; the measured speed comes from the tacho (or the accelerometer comb when the tacho is unusable). §3 says measured is about 2.3 % lower.

In [ ]:
ratio = meta["speed_ratio"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(ratio, bins=60, color="#4c72b0")
axes[0].axvline(pre.measured_speed_ratio, color="k", ls="--", lw=1, label=f"config {pre.measured_speed_ratio}")
axes[0].set(xlabel="measured / nominal shaft speed", ylabel="recordings", title="Speed ratio")
axes[0].legend()
axes[1].scatter(meta["fr_file_hz"], ratio, s=5, alpha=0.5)
axes[1].set(xlabel="nominal shaft speed (Hz)", ylabel="measured / nominal", title="Ratio vs speed")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_speed_ratio.png")
print(f"measured speed {meta.fr_measured_hz.min():.2f}–{meta.fr_measured_hz.max():.2f} Hz; ratio median "
      f"{ratio.median():.4f}, IQR {ratio.quantile(.25):.4f}–{ratio.quantile(.75):.4f}, "
      f"range {ratio.min():.4f}–{ratio.max():.4f}")

Which speed method won, and how closely the three methods agree. Tacho edges win when another method confirms them within 0.2 %; disagreement above 1 % is flagged.

In [ ]:
agree = meta["speed_agreement"] * 100
print(meta["speed_method"].value_counts().to_string())
print(f"\nagreement between methods: median {agree.median():.3f} %, 99th pct {agree.quantile(.99):.3f} %, "
      f"> 0.2 %: {(agree > 0.2).sum()}, > 1 %: {(agree > 1).sum()}")
meta.loc[meta.speed_agreement > 0.005, ["rec_id", "fr_tacho_edges", "fr_tacho_fft", "fr_accel_comb",
                                         "speed_method", "speed_agreement", "tacho_glitches"]].round(4)

### Tacho glitches
A glitch is a pulse displaced within the revolution: one revolution pair becomes a short and a long interval. Speed is computed from the clean intervals, so these files keep a correct speed. When more than 20 % of intervals are glitched, tacho edges are unusable and the tacho FFT or comb takes over.

In [ ]:
revs = meta["fr_measured_hz"] * pre.n_rows / pre.fs_hz
glitch_frac = meta["tacho_glitches"].clip(lower=0) / revs
bands = pd.cut(glitch_frac, [-1, 0, 0.05, 0.2, 10], labels=["none", "≤ 5 %", "5–20 %", "> 20 % (edges unusable)"])
pd.crosstab(meta["class10"], bands).reindex(CLASS10)

One glitched file, zoomed on its worst glitch: detected rising edges in green, the glitched interval in red.

In [ ]:
ex = meta[(meta.tacho_glitches >= 2) & (meta.speed_method == "tacho_edges")].sort_values("tacho_glitches").iloc[0]
x = np.load(data_dir() / "raw_npy" / f"{ex.rec_id}.npy", mmap_mode="r")
tacho = np.asarray(x[:, 0])
ti = tacho_intervals(tacho, pre)
period = np.median(ti.intervals)
bad = int(np.argmax(np.abs(ti.intervals / ti.revs / period - 1)))
lo, hi = ti.edges[max(bad - 2, 0)] - 500, ti.edges[min(bad + 3, ti.edges.size - 1)] + 500
t_ms = np.arange(lo, hi) / pre.fs_hz * 1e3

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(t_ms, tacho[lo:hi], lw=0.8, color="#333")
for e in ti.edges[(ti.edges >= lo) & (ti.edges < hi)]:
    ax.axvline(e / pre.fs_hz * 1e3, color="green", lw=0.8, alpha=0.7)
ax.axvspan(ti.edges[bad] / pre.fs_hz * 1e3, ti.edges[bad + 1] / pre.fs_hz * 1e3, color="red", alpha=0.12)
ax.set(xlabel="time (ms)", ylabel="tacho (V)",
       title=f"{ex.rec_id}: {int(ex.tacho_glitches)} glitched intervals, speed {ex.fr_measured_hz:.3f} Hz")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_glitch_example.png")
print("intervals around the glitch (samples):", ti.intervals[max(bad - 2, 0): bad + 3],
      "| median period:", int(period))

### Class × speed coverage
Each class should cover the full speed range, or a classifier can learn speed as a proxy for class.

In [ ]:
bins = np.arange(10, 65, 5)
speed_bin = pd.cut(meta["fr_measured_hz"], bins)
grid = pd.crosstab(meta["class10"], speed_bin).reindex(CLASS10)
fig, ax = plt.subplots(figsize=(11, 4.2))
im = ax.imshow(grid.values, aspect="auto", cmap="Blues")
ax.set_xticks(range(len(grid.columns)), [f"{int(b.left)}–{int(b.right)}" for b in grid.columns])
ax.set_yticks(range(len(grid.index)), grid.index)
for (i, j), v in np.ndenumerate(grid.values):
    ax.text(j, i, v, ha="center", va="center", fontsize=8, color="white" if v > grid.values.max() / 2 else "black")
ax.set(xlabel="measured shaft speed (Hz)", title="Recordings per class and speed band")
fig.colorbar(im, ax=ax, label="recordings")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_class_speed_heatmap.png")

### Other quality flags
Flags are for auditing only and never become model inputs. The microphone spikes need attention: they are short bursts, and they concentrate in a few classes.

In [ ]:
flags = meta["quality_flags"].fillna("").str.split(";").explode()
print(flags[flags != ""].value_counts().to_string())
spiky = meta[meta.quality_flags.fillna("").str.contains("spike_mic")]
pd.crosstab(spiky["class10"], spiky["severity"])

In [ ]:
rid = spiky.sort_values("peakz_mic").iloc[len(spiky) // 2].rec_id
mic = np.asarray(np.load(data_dir() / "raw_npy" / f"{rid}.npy", mmap_mode="r")[:, 7])
t = np.arange(mic.size) / pre.fs_hz
fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(t, mic, lw=0.4, color="#8172b2")
ax.set(xlabel="time (s)", ylabel="microphone (V)", title=f"{rid}: typical spiky microphone channel")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_mic_spikes.png")
print(f"median |mic| {np.median(np.abs(mic)):.3f} V, max |mic| {np.abs(mic).max():.1f} V")

In [ ]:
clip_cols = [c for c in meta.columns if c.startswith("rail_") and c != "rail_tacho"]
clipped = meta[meta.quality_flags.fillna("").str.contains("clip_")]
clipped[["rec_id", *clip_cols]]

In [ ]:
dc_cols = [c for c in meta.columns if c.startswith("dc_")]
meta[dc_cols].abs().describe(percentiles=[.5, .9, .99]).T[["50%", "90%", "99%", "max"]].round(3)

### Split balance
All splits are by recording and were assigned before any window exists.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
colors = {"train": "#4c72b0", "val": "#dd8452", "test": "#55a868"}
for ax, col in zip(axes, ["split_closed", "split_speed_block", "split_severity"]):
    t = pd.crosstab(meta["class10"], meta[col], normalize="index").reindex(CLASS10)[list(colors)]
    t.plot.barh(stacked=True, ax=ax, color=colors, legend=False, width=0.8)
    ax.set(title=col, xlabel="share of class recordings", ylabel="")
    ax.invert_yaxis()
axes[0].legend(loc="lower left", fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "01_split_balance.png")
pd.crosstab(meta["class10"], meta["split_closed"], normalize="index").reindex(CLASS10).round(3)

Closed-set split against speed tercile, and the LOCO columns (the withheld class never reaches train or val).

In [ ]:
display(pd.crosstab(meta["speed_tercile"], meta["split_closed"]))
pd.DataFrame({c: meta[f"loco_{c}"].value_counts() for c in FAULT_CLASSES}).T.fillna(0).astype(int)

## Findings
- **Counts match §3 exactly**: 1,951 files, all 10 classes. No shape, NaN or read problems; all 1,951 `.npy` files verified.
- **Speed**: measured 11.78–60.73 Hz. Measured/nominal has median 0.9767 (IQR 0.9752–0.9780, range 0.9688–0.9838), so §3's "about 2.3 % lower" holds. The three methods agree to a median of 0.010 %; 53 files differ by more than 0.2 % and only one by more than 1 % (`overhang__cage_fault__6g__16.1792`).
- **Tacho glitches are more common than §3 says**: 228 files have at least one displaced pulse. In 94 of them more than 20 % of intervals are glitched, so tacho edges are unusable and the tacho FFT supplies the speed. 63 of those 94 are underhang ball, which matches §3's "about 60, mostly ball faults" only for this severe group.
- **Microphone spikes are class-correlated (leakage risk)**: 164 files contain bursts up to about 160 V against a median |mic| of 0.13 V. They occur only in imbalance 20–35 g (82 files, 75 of them 30–35 g) and underhang cage 6–35 g (82 files). A model could learn the artefact instead of the fault.
- **Clipping and DC**: 9 overhang ball 6 g files at 47–55 Hz saturate the underhang accelerometers (up to 1,432 railed samples); 8 underhang cage files, mostly 35 g, rail the microphone slightly (10–17 samples). Underhang channels are near zero-mean, but the overhang accelerometers carry DC up to 2.2 V (55 flags across 3 channels) and one microphone reaches 4.5 V.
- **Splits are balanced**: `split_closed` is within 0.7 % of 70/15/15 for every class and even across speed terciles (95/95/100 test). `split_speed_block` puts 776 recordings (40 %) in test; `split_severity` puts 370 in test. Overhang ball thins out above 35 Hz (4–11 recordings per 5 Hz band against about 20 elsewhere), so it has the smallest speed-block test share (36 of 137, 26 %).


## Outputs written
- `results/figures/01_speed_ratio.png`
- `results/figures/01_glitch_example.png`
- `results/figures/01_class_speed_heatmap.png`
- `results/figures/01_mic_spikes.png`
- `results/figures/01_split_balance.png`

`metadata.csv` and `data/raw_npy/` are produced by the scripts, not by this notebook.